<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/09-miniproyecto-pronostico-cartera/ejercicio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Ejercicio 1.9 · Miniproyecto: el pronóstico de la cartera

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

Antes de empezar, guarda tu copia: **Archivo → Guardar una copia en Drive**.

Escribe tu respuesta en cada celda de código y ejecútala con **Shift + Enter**.

### Prepara los datos

Ejecuta estas seis celdas. Traen la limpieza, la tabla y `pronosticar_cartera`. Deben mostrar `50000` ventas y `13582286.61`.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [ ]:
def limpiar_monto(columna):
    return columna.str.replace("S/ ", "").str.replace(",", "").astype(float)


def limpiar_ventas(archivo):
    ventas = pd.read_csv(archivo, sep=";")

    # Nombres de columna
    ventas.columns = ventas.columns.str.lower().str.replace(" ", "_")
    ventas = ventas.rename(
        columns={
            "nro_venta": "id_venta",
            "cliente": "id_cliente",
            "producto": "id_producto",
            "vendedor": "id_vendedor",
            "método_de_pago": "metodo_pago",
        }
    )

    # Duplicados
    ventas = ventas.drop_duplicates()

    # Textos
    ventas["ciudad"] = ventas["ciudad"].str.strip().str.capitalize()
    ventas["metodo_pago"] = ventas["metodo_pago"].str.strip().str.capitalize()
    ventas["metodo_pago"] = ventas["metodo_pago"].replace(
        {"Yape": "Billetera digital", "Plin": "Billetera digital"}
    )
    ventas["metodo_pago"] = ventas["metodo_pago"].fillna("Sin dato")

    # Números guardados como texto
    ventas["precio_unitario"] = limpiar_monto(ventas["precio_unitario"])
    ventas["total"] = limpiar_monto(ventas["total"])
    ventas["descuento"] = ventas["descuento"].str.replace("%", "").astype(float) / 100

    # Fechas
    texto = ventas["fecha"]
    ventas["fecha"] = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    ventas["fecha"] = ventas["fecha"].fillna(
        pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce")
    )
    ventas["anio"] = ventas["fecha"].dt.year
    ventas["mes"] = ventas["fecha"].dt.month

    # Cantidades vacías o con errores de tipeo
    cantidad = ventas["total"] / (ventas["precio_unitario"] * (1 - ventas["descuento"]))
    ventas["cantidad"] = cantidad.round().astype(int)

    return ventas

In [ ]:
def tabla_de_pronostico(ventas, clientes):
    """Una fila por cliente y mes, con cero si no compró, y lo que se sabía antes de ese mes."""
    meses = ventas["fecha"].str[:7].rename("mes")
    todos = pd.MultiIndex.from_product(
        [clientes["id_cliente"], sorted(meses.unique())], names=["id_cliente", "mes"]
    )
    compras = ventas.groupby(["id_cliente", meses])["total"].sum()
    tabla = compras.reindex(todos, fill_value=0).reset_index()

    por_cliente = tabla.groupby("id_cliente")["total"]
    tabla["mes_anterior"] = por_cliente.shift(1)
    tabla["promedio_3"] = por_cliente.transform(lambda s: s.shift(1).rolling(3).mean())
    tabla["promedio_6"] = por_cliente.transform(lambda s: s.shift(1).rolling(6).mean())
    tabla["numero_mes"] = tabla["mes"].str[5:].astype(int)

    columnas = ["id_cliente", "tipo", "ciudad", "id_vendedor", "fecha_alta"]
    tabla = pd.merge(tabla, clientes[columnas], on="id_cliente", how="left")
    tabla = tabla.dropna(subset=["promedio_6"])
    return tabla[tabla["mes"] + "-01" >= tabla["fecha_alta"]].drop(columns="fecha_alta")

In [ ]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [ ]:
def pronosticar_cartera(tabla, indice, mes):
    """Entrena con lo anterior a `mes` y compara tres métodos por cartera de vendedor."""
    entreno = tabla[tabla["mes"] < mes]
    objetivo = tabla[tabla["mes"] == mes]

    numeros = ["mes_anterior", "promedio_3"]
    textos = ["tipo", "ciudad", "numero_mes"]
    x_entreno = preparar_x(entreno, numeros, textos)
    x_objetivo = preparar_x(objetivo, numeros, textos, x_entreno.columns)

    modelo = LinearRegression()
    modelo.fit(x_entreno, entreno["total"])

    resultado = objetivo.assign(
        regresion=modelo.predict(x_objetivo),
        regla_indice=objetivo["promedio_6"] * objetivo["numero_mes"].map(indice),
    )
    columnas = ["total", "promedio_6", "regla_indice", "regresion"]
    return resultado.groupby("id_vendedor")[columnas].sum().round(2)

In [ ]:
ventas = limpiar_ventas(DATOS + "ventas_crudo.csv")
ventas["fecha"] = ventas["fecha"].dt.strftime("%Y-%m-%d")
clientes = pd.read_csv(DATOS + "clientes.csv")

tabla = tabla_de_pronostico(ventas, clientes)

ventas_2024 = ventas[ventas["fecha"].str[:4] == "2024"]
por_mes = ventas_2024.groupby(ventas_2024["fecha"].str[5:7].astype(int))["total"].sum()
indice = por_mes / por_mes.mean()

print(len(ventas), round(ventas["total"].sum(), 2))
print(len(tabla))

### 1. Otro mes

Haz de cuenta que hoy es 31 de octubre. Pronostica noviembre con `pronosticar_cartera(tabla, indice, "2025-11")` y guárdalo en `noviembre`. Calcula el error medio de los tres métodos. Deben dar `5287`, `4080` y `4777`.

¿Quién gana este mes? ¿Qué te dice eso, sumado a lo de diciembre? Responde en un comentario.

In [ ]:
# Escribe tu código aquí

### 2. Un mes sin picos

Repite con octubre (`"2025-10"`). Deben dar `2611`, `2648` y `2053`. ¿Ayuda el índice en un mes normal? Responde en un comentario.

In [ ]:
# Escribe tu código aquí

### 3. El total de la empresa

Suma las columnas de `noviembre` con `.sum()`. El total real debe dar `642484` aproximadamente. ¿Cuánto se aleja cada método, en porcentaje?

In [ ]:
# Escribe tu código aquí

### 4. Los clientes perdidos

`datos/README.md` pregunta si podrás encontrarlos. Busca la última compra de cada cliente, con `ventas.groupby("id_cliente")["fecha"].max()`, y quédate con los que no compraron desde el 1 de octubre de 2025. Deben ser `30`. Para esta lista miramos todos los datos, hasta fin de 2025.

Guárdalos en `perdidos`, ordenados de la última compra más antigua a la más reciente, y muestra los primeros 5. El primero debe ser `C0090`.

> 💡 Las fechas son texto `aaaa-mm-dd`, así que se pueden comparar con `<`.

In [ ]:
# Escribe tu código aquí

### 5. Tu reporte

Guarda en `reporte_cartera.xlsx` dos hojas: `Cartera`, con `noviembre`, y `Perdidos`, con `perdidos`. Vuelve a abrir la hoja `Cartera` y comprueba que la suma de `total` sea la misma que en el ejercicio 3.

> 💡 Como en la clase: usa `.reset_index()` antes de `.to_excel()`, para no perder los códigos.

In [ ]:
# Escribe tu código aquí

### 6. La nota para gerencia

Escribe en un comentario, en 3 líneas: qué método recomiendas para el próximo mes, qué error esperar, y para qué **no** debe usarse. Justifícalo con cifras de los ejercicios.

In [ ]:
# Escribe tu código aquí

## 🎉 ¡Terminaste!

Compara tus respuestas con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/09-miniproyecto-pronostico-cartera/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0